In [1]:
# ---------- Cell 6 (PPMI 定制版) : 绘制并导出精美脉络化全局 CustomKG 图谱 ----------
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from networkx.algorithms import community
import os
import math

# 1. 解决中文乱码与报错问题
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False 

# 配置 PPMI 文件路径
PREFIX = "distmult_PPMI"
TSV_FILES = [f"train_{PREFIX}.tsv", f"valid_{PREFIX}.tsv", f"test_{PREFIX}.tsv"]

# 2. 统计变量初始化 (针对 PPMI 特征精细化分类)
patients = set()
demo_nodes = set()  # 人口学节点 (age, sex)
med_nodes = set()   # 临床与用药状态节点 (EHR部分)
phe_nodes = set()   # 临床表型量表节点 (PHE部分, 对应原bio/test)
edges = []

count_demo = 0
count_med = 0
count_phe = 0

# PPMI 专属关系分类规则
demo_rels = {'has_age_bin', 'has_sex'}
med_rels = {'has_upd23a_medication_for_pd', 'has_upd23b_clinical_state_on_medication', 'has_upd2da_dyskinesias_during_exam', 'has_upd2db_movements_interfere_with_ratings'}

# 3. 读取数据并进行精确分类统计
print(">> 读取图谱文件并分类节点与关系...")
for file in TSV_FILES:
    if os.path.exists(file):
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) == 3:
                    h, r, t = parts
                    patients.add(h)
                    edges.append((h, t))
                    
                    # 针对 PPMI 数据集的特征进行同步分类
                    if r in demo_rels:
                        count_demo += 1
                        demo_nodes.add(t)
                    elif r in med_rels:
                        count_med += 1
                        med_nodes.add(t)
                    else:
                        count_phe += 1
                        phe_nodes.add(t)

# 合并所有特征节点以计算总数
all_features = demo_nodes | med_nodes | phe_nodes
total_triples = len(edges)
total_nodes = len(patients) + len(all_features)

print(f"✅ 数据读取完毕！总节点: {total_nodes}, 总三元组/边: {total_triples}")
print(">> 正在构建 NetworkX 图结构...")

# 4. 构建 NetworkX 图
G = nx.Graph()
G.add_nodes_from(patients)
G.add_nodes_from(all_features)
G.add_edges_from(edges)

# 5. 绘图配置与脉络化布局方案
plt.figure(figsize=(24, 24), dpi=300) 
ax = plt.gca()
ax.set_facecolor('#ffffff') 

print(">> 正在进行社区检测以发现潜在脉络...")
communities_generator = community.greedy_modularity_communities(G)
communities = sorted(communities_generator, key=len, reverse=True)
num_communities = len(communities)
print(f"✅ 发现 {num_communities} 个主要社区岛屿！")

# 独立布局社区中心
meta_pos = {}
radius = 12.0 
for i in range(num_communities):
    angle = (2 * math.pi * i) / num_communities
    meta_pos[i] = (radius * math.cos(angle), radius * math.sin(angle))

# 为每个社区内的节点进行局部布局
pos = {}
print(">> 正在为各个社区岛屿内部节点进行布局...")
for i, comm in enumerate(communities):
    subgraph = G.subgraph(comm)
    sub_pos = nx.spring_layout(subgraph, k=0.1, iterations=15, seed=42, center=meta_pos[i], scale=2.0)
    for node, (x, y) in sub_pos.items():
        pos[node] = (x, y)

print("✅ 基于社区的脉络化布局计算完成。")

# 6. 渲染图形
print(">> 正在渲染精美分类节点与曲线连线...")
nx.draw_networkx_edges(G, pos, width=0.06, alpha=0.15, edge_color='#999999', connectionstyle='arc3,rad=0.05')

# 替换分类节点颜色渲染 (适配 PPMI)
nx.draw_networkx_nodes(G, pos, nodelist=list(phe_nodes), node_color='#2196F3', node_size=18, alpha=0.98, edgecolors='none')  # 蓝色：临床表型(PHE)
nx.draw_networkx_nodes(G, pos, nodelist=list(med_nodes), node_color='#9C27B0', node_size=18, alpha=0.98, edgecolors='none')   # 紫色：临床与用药(EHR)
nx.draw_networkx_nodes(G, pos, nodelist=list(demo_nodes), node_color='#FFC107', node_size=18, alpha=0.98, edgecolors='none')  # 琥珀黄：人口学(EHR)
nx.draw_networkx_nodes(G, pos, nodelist=list(patients), node_color='#FF4B4B', node_size=12, alpha=0.98, edgecolors='none')       # 珊瑚红：患者

# 7. 统计信息框与图例
stats_text = (
    f"CustomKG Global Topology (PPMI)\n"
    f"─脉络化社区布局(Community-based)─\n"
    f"Total Nodes (节点总数): {total_nodes:,}\n"
    f"  - Patients: {len(patients):,}\n"
    f"  - Features: {len(all_features):,}\n"
    f"Total Triples (三元组总数): {total_triples:,}\n\n"
    f"Relation & Feature Breakdown (关系与节点分解):\n"
    f" - Demographics (人口学特征): {count_demo:,} edges\n"
    f" - Clinical & Medication (临床与用药状态): {count_med:,} edges\n"
    f" - Phenotype Tests (临床表型量表): {count_phe:,} edges"
)

props = dict(boxstyle='round', facecolor='#F8F9FA', alpha=0.98, edgecolor='#CCCCCC')
ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=14,
        verticalalignment='top', bbox=props, fontweight='bold')

legend_handles = [
    mpatches.Patch(color='#FF4B4B', label='Patient Nodes (患者核心实体)'),
    mpatches.Patch(color='#FFC107', label='Demographics (人口学特征)'),
    mpatches.Patch(color='#9C27B0', label='Clinical & Medication (临床与用药状态)'),
    mpatches.Patch(color='#2196F3', label='Phenotype Tests (临床表型量表)')
]
plt.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.02, 0.81), 
           fontsize=14, framealpha=0.98, edgecolor='#CCCCCC')

plt.axis('off')
plt.tight_layout()

# 8. 导出 PDF
output_pdf = "PPMI-CustomKGE.pdf"
print(f">> 正在导出矢量 PDF 至 {output_pdf} ...")
plt.savefig(output_pdf, format='pdf', bbox_inches='tight')
plt.close()
print(f"✅ 导出完成！你可以去查看 PPMI 专属的图谱文件 {output_pdf} 了。")

>> 读取图谱文件并分类节点与关系...
✅ 数据读取完毕！总节点: 677, 总三元组/边: 16410
>> 正在构建 NetworkX 图结构...
>> 正在进行社区检测以发现潜在脉络...
✅ 发现 22 个主要社区岛屿！
>> 正在为各个社区岛屿内部节点进行布局...
✅ 基于社区的脉络化布局计算完成。
>> 正在渲染精美分类节点与曲线连线...
>> 正在导出矢量 PDF 至 PPMI-CustomKGE.pdf ...


C:\Users\HP\AppData\Local\Temp\ipykernel_10600\784937619.py:100: UserWarning: 

The connectionstyle keyword argument is not applicable when drawing edges
with LineCollection.

To make this warning go away, either specify `arrows=True` to
force FancyArrowPatches or use the default values.
Note that using FancyArrowPatches may be slow for large graphs.

  nx.draw_networkx_edges(G, pos, width=0.06, alpha=0.15, edge_color='#999999', connectionstyle='arc3,rad=0.05')


✅ 导出完成！你可以去查看 PPMI 专属的图谱文件 PPMI-CustomKGE.pdf 了。


In [1]:
import networkx as nx
from pyvis.network import Network
import os
import random
import warnings

warnings.filterwarnings('ignore')

# 配置和文件路径
PREFIX = "distmult_PPMI"
TSV_FILES = [f"train_{PREFIX}.tsv", f"valid_{PREFIX}.tsv", f"test_{PREFIX}.tsv"]

# ⚠️ 注意：由于使用了 Pyvis 交互式图谱，输出格式需要更改为 HTML，在此向您确认该文件名是否可行。
OUTPUT_HTML_PATH = "PPMI-CustomKG-Topology.html"

def plot_ppmi_custom_kg():
    print("==================================================")
    print("🌐 开始构建多类型均衡的 PPMI-CustomKG 连通图谱 (白色背景, 高对比度配色)...")

    # 1. 初始化统计映射与基础集合
    node_type_map = {}
    edges_data = []
    count_demo = 0
    count_med = 0
    count_phe = 0

    # PPMI 专属关系分类规则
    demo_rels = {'has_age_bin', 'has_sex'}
    med_rels = {'has_upd23a_medication_for_pd', 'has_upd23b_clinical_state_on_medication', 'has_upd2da_dyskinesias_during_exam', 'has_upd2db_movements_interfere_with_ratings'}

    # 2. 读取数据并进行精确分类统计
    print(f">> 正在读取 {PREFIX} 图谱文件并分类节点与关系...")
    for file in TSV_FILES:
        if os.path.exists(file):
            with open(file, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split('\t')
                    if len(parts) == 3:
                        h, r, t = parts
                        # 头实体固定为患者
                        node_type_map[h] = 'patient'
                        edges_data.append((h, t, r))
                        
                        # 关系与尾实体(特征)同步分类
                        if r in demo_rels:
                            node_type_map[t] = 'demographics'
                            count_demo += 1
                        elif r in med_rels:
                            node_type_map[t] = 'clinical_medication'
                            count_med += 1
                        else:
                            node_type_map[t] = 'phenotype_tests'
                            count_phe += 1

    if not edges_data:
        print("❌ 未读取到任何三元组数据，请检查 TSV 文件。")
        return

    total_triples = len(edges_data)
    total_nodes = len(node_type_map)
    print(f"✅ PPMI 数据读取完毕！总节点: {total_nodes}, 总三元组/边: {total_triples}")

    # 3. 构建全量无向图用于寻径
    print(">> 正在构建 NetworkX 图结构...")
    G_full = nx.Graph()
    for h, t, r in edges_data:
        G_full.add_edge(h, t, relation=r)

    # 4. 提取多类型均衡连通子图 (Shortest Path Routing)
    print("⏳ 正在进行多类型路由寻径，确保包含所有类型节点且不断开...")
    largest_cc = max(nx.connected_components(G_full), key=len)
    G_sub = G_full.subgraph(largest_cc)

    central_node = sorted(G_sub.degree, key=lambda x: x[1], reverse=True)[0][0]

    nodes_by_type = {}
    for node in G_sub.nodes():
        ntype = node_type_map.get(node, 'unknown')
        if ntype not in nodes_by_type:
            nodes_by_type[ntype] = []
        nodes_by_type[ntype].append(node)

    random.seed(42) 
    TARGET_PER_TYPE = 60
    core_nodes = set([central_node])

    for ntype, nodes_list in nodes_by_type.items():
        sampled_targets = random.sample(nodes_list, min(TARGET_PER_TYPE, len(nodes_list)))
        for target in sampled_targets:
            try:
                path = nx.shortest_path(G_sub, source=central_node, target=target)
                core_nodes.update(path)
            except nx.NetworkXNoPath:
                continue

    G_core = G_sub.subgraph(core_nodes)

    # 5. 高对比度配色方案与名称映射
    print(">> 正在渲染精美分类节点与曲线连线...")
    color_map = {
        'patient': '#FF4B4B',               # 患者实体
        'demographics': '#FFC107',          # 人口学特征
        'clinical_medication': '#9C27B0',   # 临床与用药状态
        'phenotype_tests': '#2196F3'        # 临床表型量表
    }

    type_name_map = {
        'patient': 'Patient Nodes (患者核心实体)',
        'demographics': 'Demographics (人口学特征)',
        'clinical_medication': 'Clinical & Medication (临床与用药状态)',
        'phenotype_tests': 'Phenotype Tests (临床表型量表)'
    }

    net = Network(height="900px", width="100%", bgcolor="#ffffff", font_color="#333333", select_menu=True)

    # 注入节点
    for node in G_core.nodes():
        ntype = node_type_map.get(node, 'unknown')
        color = color_map.get(ntype, '#A9A9A9')
        display_type = type_name_map.get(ntype, ntype)
        
        # 区分患者与特征节点的尺寸
        node_size = 15 if ntype != 'patient' else 10

        net.add_node(node, 
                     label=" ", 
                     title=f"实体名称: {node}\n实体类型: {display_type}",
                     color=color,
                     size=node_size)

    # 注入连线
    for u, v, data in G_core.edges(data=True):
        net.add_edge(u, v, 
                     title=data.get('relation', ''), 
                     label=' ', 
                     width=0.5,         
                     color='#bdc3c7')   

    net.force_atlas_2based(gravity=-50, central_gravity=0.01, spring_length=150, spring_strength=0.05, damping=0.4, overlap=0)
    net.write_html(OUTPUT_HTML_PATH)

    # 6. 注入全局统计面板
    patient_count = sum(1 for v in node_type_map.values() if v == 'patient')
    feature_count = total_nodes - patient_count

    legend_html = f"""
    <div style="position: absolute; top: 20px; left: 20px; z-index: 999; background-color: rgba(255, 255, 255, 0.95); color: #333; padding: 15px 25px; border-radius: 8px; border: 1px solid #ddd; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
        <h3 style="margin-top: 0; margin-bottom: 10px; font-size: 16px; border-bottom: 2px solid #eee; padding-bottom: 5px; color: #2c3e50;">CustomKG Global Topology (PPMI)</h3>
        <p style="font-size: 12px; color: #666; margin-top: 0; margin-bottom: 15px;">─ 基于诊断表 PPMI 构建的私域知识图谱 ─</p>
        
        <div style="font-size: 13px; margin-bottom: 15px; border-bottom: 1px solid #eee; padding-bottom: 10px;">
            <div><b>Total Nodes:</b> {total_nodes:,}</div>
            <div style="margin-left: 10px; color: #555;">- Patients: {patient_count:,}</div>
            <div style="margin-left: 10px; color: #555;">- Features: {feature_count:,}</div>
            <div style="margin-top: 5px;"><b>Total Triples:</b> {total_triples:,}</div>
        </div>
        
        <h4 style="margin: 0 0 10px 0; font-size: 14px;">Module Breakdown:</h4>
    """
    
    # 按照原设定分类展示边数与图例
    stats_data = [
        ('demographics', count_demo),
        ('clinical_medication', count_med),
        ('phenotype_tests', count_phe)
    ]

    for ntype, count in stats_data:
        color = color_map.get(ntype, '#A9A9A9')
        display_name = type_name_map.get(ntype, ntype)
        legend_html += f'<div style="margin: 8px 0; font-size: 13px; display: flex; align-items: center;"><span style="display: inline-block; width: 14px; height: 14px; background-color: {color}; border-radius: 50%; margin-right: 12px; border: 1px solid #888;"></span><span style="font-weight: 500; width: 250px;">{display_name}</span> <span>{count:,} edges</span></div>'
    
    # 单独展示 Patient 颜色图示
    patient_color = color_map['patient']
    legend_html += f'<div style="margin: 8px 0; font-size: 13px; display: flex; align-items: center;"><span style="display: inline-block; width: 14px; height: 14px; background-color: {patient_color}; border-radius: 50%; margin-right: 12px; border: 1px solid #888;"></span><span style="font-weight: 500; width: 250px;">{type_name_map["patient"]}</span></div>'

    legend_html += "</div>"

    with open(OUTPUT_HTML_PATH, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    html_content = html_content.replace('<body>', f'<body>\n{legend_html}')
    
    with open(OUTPUT_HTML_PATH, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"💾 PPMI 私域知识图谱拓扑已生成，文件保存至: {OUTPUT_HTML_PATH}")
    print("==================================================")

if __name__ == "__main__":
    plot_ppmi_custom_kg()

🌐 开始构建多类型均衡的 PPMI-CustomKG 连通图谱 (白色背景, 高对比度配色)...
>> 正在读取 distmult_PPMI 图谱文件并分类节点与关系...
✅ PPMI 数据读取完毕！总节点: 677, 总三元组/边: 16410
>> 正在构建 NetworkX 图结构...
⏳ 正在进行多类型路由寻径，确保包含所有类型节点且不断开...
>> 正在渲染精美分类节点与曲线连线...
💾 PPMI 私域知识图谱拓扑已生成，文件保存至: PPMI-CustomKG-Topology.html
